In [2]:
%load_ext autoreload
%autoreload 2

import time
import random
from copy import deepcopy
from itertools import tee
from typing import Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch as th
from torch.utils.data import DataLoader
from torch.nn import functional as F
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from toolz.curried import *
from toolz.sandbox.core import unzip

from mmidas.nn_model import make_mmidas
from mmidas.model import Net
from mmidas.train import generic_train
from mmidas._data_util import make_loaders, viz, make_mnist
from mmidas._utils import randomize, shuffle, npercent

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
train_loader, val_loader, test_loader = make_loaders('mnist', batch_size=32)
train_mnist, val_mnist, test_mnist = make_mnist(datasets.MNIST) # batch with partition()

In [ ]:
net = Net()
opt = th.optim.SGD(net.parameters(), lr=1e-3)
metrics = generic_train(net, opt, train_loader, val_loader, epochs=10)

In [36]:
device = 'cpu'
n_categories = 10
state_dim = 2
input_dim = 784
n_arm = 2
lr = 1e-3

model = make_mmidas(10, 2, 784, n_arm=n_arm, device=device).to(device)
opt = th.optim.Adam(model.parameters(), lr=1e-3)

In [66]:
def simplify_mmidas(f):
    def g(x):
        return random.choice(f([x for _ in range(f.n_arm)], temp=1)[4])
    return g

In [68]:
x, y = first(train_loader)

simplify_mmidas(model)(x)

tensor([[1.9582e-18, 3.8284e-17, 2.1275e-17, 9.0626e-19, 1.0000e+00, 2.0470e-14,
         2.9926e-13, 6.1017e-23, 3.0280e-20, 1.2224e-18],
        [1.9521e-23, 1.4836e-23, 1.3380e-23, 7.8192e-24, 1.0000e+00, 4.7370e-20,
         3.0569e-19, 7.6666e-29, 1.9866e-25, 1.0006e-24],
        [1.9523e-21, 1.4838e-19, 5.8266e-20, 1.7354e-18, 4.4253e-18, 5.0902e-21,
         3.9308e-19, 9.9956e-04, 1.5635e-17, 9.9900e-01],
        [3.2972e-25, 3.1015e-20, 3.7530e-20, 1.1581e-20, 6.2517e-21, 9.1760e-21,
         1.5612e-20, 1.0000e+00, 4.1497e-23, 5.1875e-12],
        [3.0681e-21, 3.4264e-17, 5.4431e-17, 1.5435e-17, 4.5204e-17, 5.8558e-18,
         5.3669e-17, 1.0000e+00, 9.2451e-19, 2.1084e-06],
        [1.3682e-25, 2.3258e-38, 1.9387e-35, 3.5778e-32, 9.9844e-33, 3.6329e-34,
         5.1153e-37, 1.4258e-36, 1.0000e+00, 2.5292e-33],
        [1.2500e-11, 1.0056e-10, 1.0982e-06, 1.8489e-08, 7.3767e-01, 2.6233e-01,
         7.9004e-08, 4.8413e-11, 2.8758e-09, 1.8488e-07],
        [2.3363e-17, 7.9308

In [ ]:
metrics = generic_train(model, opt, train_loader, val_loader, 1000, F.cross_entropy, device, simplify_mmidas)

In [ ]:
x_recs, _, _, x_lows, cs, s_smps, c_smps, s_means, s_logvars, c_probs = model([x for _ in range(n_arm)], 1)
cs = th.stack(cs)

th.argmax(cs, dim=-1)

In [ ]:
tic = time.time()
loss_naive = model.loss_naive(cs)
t1 = time.time() - tic

tic = time.time()
loss_vec = model.loss_vectorize(cs)
t2 = time.time() - tic

print(f"Naive loss: {loss_naive.item()}")
print(f"Vectorized loss: {loss_vec.item()}")
print(f"Relative error: {th.norm(loss_naive - loss_vec) / th.norm(loss_naive)}\n")


print(f"Naive loss computation took: {t1}s")
print(f"Vectorized loss computation took: {t2}s")
print(f"Speedup: {100 * (t1 - t2) / t1:.2f}%")